# Evaluasi Perbandingan Model

Notebook ini akan melakukan evaluasi komparatif untuk 4 skenario pemodelan dari notebook sebelumnya:
1. **SVM Tanpa Augmentasi**
2. **SVM Dengan Augmentasi**
3. **MobileNetV2 Tanpa Augmentasi**
4. **MobileNetV2 Dengan Augmentasi**

Evaluasi dilakukan menggunakan `classification_report` (akurasi, precision, recall, f1-score) serta visualisasi Confusion Matrix dan grafik perbandingan performa antar skenario.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
from tensorflow.keras.models import load_model
import pandas as pd
from pathlib import Path

CLASSES = ["WithMask", "WithoutMask", "MaskWornIncorrect"]
plt.rcParams['figure.figsize'] = (8, 6)

## 1. Fungsi Visualisasi

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
    plt.title(title, pad=15)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

def evaluate_scenario(y_true, y_pred, scenario_name):
    print(f"\n{'='*50}")
    print(f"SCENARIO: {scenario_name}")
    print(f"{'='*50}")
    # Print Classification Report
    report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=False)
    print(report)
    
    # Dapatkan metrics dari report (bentuk dict) untuk charting nanti
    report_dict = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
    
    plot_confusion_matrix(y_true, y_pred, title=f"Confusion Matrix: {scenario_name}")
    return report_dict

## 2. Prediksi dan Evaluasi Model
Pastikan model sudah disave di `../models/` dan data test (Test/Validation) tersedia.

In [ ]:
# Dictionary untuk menyimpan report semua skenario
all_reports = {}

# ---------------------------------------------------------
# A. Skenario SVM (Load Fitur Canny Unaugmented sebagai target test)
# ---------------------------------------------------------
print("Loading SVM Test Features...")
data_unaug = np.load('../data/features/canny_unaug_features.npz')
X_val_svm = data_unaug['X_val']
y_val_svm = data_unaug['y_val']

# Skenario 1
svm_unaug = joblib.load('../models/svm_unaug.pkl')
pred_svm_unaug = svm_unaug.predict(X_val_svm)
all_reports['SVM (No Aug)'] = evaluate_scenario(y_val_svm, pred_svm_unaug, "SVM Tanpa Augmentasi")

# Skenario 2
svm_aug = joblib.load('../models/svm_aug.pkl')
pred_svm_aug = svm_aug.predict(X_val_svm)
all_reports['SVM (Augmented)'] = evaluate_scenario(y_val_svm, pred_svm_aug, "SVM Dengan Augmentasi")

# ---------------------------------------------------------
# B. Skenario MobileNetV2
# ---------------------------------------------------------
print("Loading MobileNet Test Generator...")
# Konfigurasi generator harus sama (tanpa aug) untuk set validasi
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import cv2

def mobilenet_preprocessing(img_array):
    img = img_array.astype(np.uint8)
    if img.shape[-1] == 3: img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img = clahe.apply(img)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    img_3_channel = np.stack((img,)*3, axis=-1)
    return preprocess_input(img_3_channel.astype(np.float32))

val_gen = ImageDataGenerator(preprocessing_function=mobilenet_preprocessing).flow_from_directory(
    "../data/raw/Face Mask Dataset/Validation", 
    target_size=(224, 224), 
    batch_size=32, 
    class_mode='categorical',
    shuffle=False # Sangat penting untuk match urutan label dengan prediksinya!
)
y_val_cnn = val_gen.classes

# Skenario 3
mobilenet_unaug = load_model('../models/mobilenet_unaug.h5')
pred_cnn_unaug_prob = mobilenet_unaug.predict(val_gen)
pred_cnn_unaug = np.argmax(pred_cnn_unaug_prob, axis=1)
all_reports['MobileNetV2 (No Aug)'] = evaluate_scenario(y_val_cnn, pred_cnn_unaug, "MobileNetV2 Tanpa Augmentasi")

# Skenario 4
mobilenet_aug = load_model('../models/mobilenet_aug.h5')
pred_cnn_aug_prob = mobilenet_aug.predict(val_gen)
pred_cnn_aug = np.argmax(pred_cnn_aug_prob, axis=1)
all_reports['MobileNetV2 (Augmented)'] = evaluate_scenario(y_val_cnn, pred_cnn_aug, "MobileNetV2 Dengan Augmentasi")


## 3. Komparasi Performa Skenario
Membuat Bar Chart yang membandingkan Metrik (Akurasi & Macro Avg F1-Score) antar skenario.

In [ ]:
scenarios = list(all_reports.keys())
accuracies = [all_reports[s]['accuracy'] for s in scenarios]
f1_scores = [all_reports[s]['macro avg']['f1-score'] for s in scenarios]

x = np.arange(len(scenarios))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#4C78A8')
rects2 = ax.bar(x + width/2, f1_scores, width, label='Macro F1-Score', color='#F58518')

ax.set_ylabel('Scores')
ax.set_title('Perbandingan Performa 4 Skenario')
ax.set_xticks(x)
ax.set_xticklabels(scenarios, rotation=15)
ax.set_ylim(0, 1.1)  # Biar ada ruang untuk label di atas bar
ax.legend()

def autolabel(rects):
    """Menambahkan label teks (nilai) di atas bar."""
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()
